# 02 — Chất lượng dữ liệu, cấu trúc Roster, Chronology và Khóa Split

**Mục tiêu:** Audit trùng lặp, kiểm tra roster, phân cấp Chronology Grade (A/B/C) và khóa Split Manifest trước khi phân tích quan hệ outcome (Gate G2).

Single Source of Truth: `PUBG_RESEARCH_SPEC.md` v3.0 | `PUBG_IMPLEMENTATION_PLAN.md`


Chọn `runtime` để chạy không cần Drive, hoặc `drive` để 13 notebook dùng chung dữ liệu bền vững. Với `drive`, mọi notebook phải dùng cùng `PUBG_DRIVE_PROJECT_ROOT` và chạy theo thứ tự.


In [ ]:
# @title Chọn nơi lưu dữ liệu { display-mode: "form" }
# @markdown `runtime`: không cần Drive, phù hợp notebook All-in-One.
# @markdown `drive`: lưu nối tiếp 13 notebook trong cùng thư mục Google Drive.
PUBG_STORAGE_MODE = "runtime"  # @param ["runtime", "drive"]
PUBG_DRIVE_PROJECT_ROOT = "/content/drive/MyDrive/Project_PUBG"  # @param {type:"string"}


In [ ]:
# Bootstrap: runtime mode needs no Drive; drive mode persists stage outputs.
import base64
import importlib.util
import io
import os
from pathlib import Path
import subprocess
import sys
import zipfile

IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
PUBG_STORAGE_MODE = globals().get("PUBG_STORAGE_MODE", "runtime").strip().lower()
if PUBG_STORAGE_MODE not in {"runtime", "drive"}:
    raise ValueError("PUBG_STORAGE_MODE must be 'runtime' or 'drive'")

if PUBG_STORAGE_MODE == "drive":
    if not IN_COLAB:
        raise RuntimeError("Drive mode is available only on Google Colab")
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path(globals().get(
        "PUBG_DRIVE_PROJECT_ROOT", "/content/drive/MyDrive/Project_PUBG"
    )).expanduser().resolve()
else:
    _candidates = [Path.cwd(), *Path.cwd().parents, Path("/content/Project_PUBG")]
    _candidates += [p / "Project_PUBG" for p in list(_candidates)]
    PROJECT_ROOT = next((p.resolve() for p in _candidates
                         if (p / "configs/data.yaml").is_file() and (p / "src/utils/config.py").is_file()), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = (Path("/content") if IN_COLAB else Path.cwd()) / "Project_PUBG"

if not (PROJECT_ROOT / "configs/data.yaml").is_file() or not (PROJECT_ROOT / "src/utils/config.py").is_file():
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    _bundle = zipfile.ZipFile(io.BytesIO(base64.b64decode('...')))
    for _entry in _bundle.infolist():
        _target = (PROJECT_ROOT / _entry.filename).resolve()
        if not _target.is_relative_to(PROJECT_ROOT.resolve()):
            raise ValueError("Invalid bundled path")
        if not _target.exists():
            _target.parent.mkdir(parents=True, exist_ok=True)
            _target.write_bytes(_bundle.read(_entry))
    _bundle.close()

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if globals().get("PUBG_INSTALL_DEPENDENCIES", IN_COLAB) and not globals().get("_PUBG_PACKAGES_READY", False):
    _requirements = {
        "numpy": "numpy>=1.24.0", "pandas": "pandas>=2.0.0",
        "pyarrow": "pyarrow>=12.0.0", "duckdb": "duckdb>=0.9.0",
        "scipy": "scipy>=1.10.0", "sklearn": "scikit-learn>=1.3.0",
        "yaml": "pyyaml>=6.0",
    }
    _missing = [spec for module, spec in _requirements.items() if importlib.util.find_spec(module) is None]
    if _missing:
        print("Installing missing packages:", ", ".join(_missing))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--prefer-binary", *_missing])
    _PUBG_PACKAGES_READY = True

from src.utils.config import load_config, resolve_paths
cfg = load_config(str(PROJECT_ROOT / "configs"))
if PUBG_STORAGE_MODE == "drive":
    cfg["paths"]["environments"]["drive"] = {
        "raw_root": str(PROJECT_ROOT / "data/raw"),
        "data_root": str(PROJECT_ROOT / "data"),
        "artifacts_root": str(PROJECT_ROOT / "artifacts"),
        "figures_root": str(PROJECT_ROOT / "figures"),
        "reports_root": str(PROJECT_ROOT / "reports"),
        "temp_dir": globals().get("PUBG_RUNTIME_TEMP_DIR", "/content/temp"),
    }
    cfg["paths"]["active_environment"] = "drive"
paths = resolve_paths(cfg)
for _path in paths.values():
    _path.mkdir(parents=True, exist_ok=True)
print("Project:", PROJECT_ROOT)
print("Storage:", paths["data_root"], "| Results:", paths["reports_root"])
if PUBG_STORAGE_MODE == "drive":
    print("Storage mode: Google Drive. Stage outputs persist for the next notebook.")
else:
    print("Storage mode: runtime. No Drive authorization required; export before reset.")


In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()  # bootstrap has located the project and set cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import load_config, resolve_paths
from src.data.io import get_duckdb_connection, atomic_write_json
from src.data.cleaning import audit_and_clean_aggregate_data
from src.data.match_metadata import build_match_metadata
from src.models.splits import create_split_assignments
from src.analysis.eda import run_chronology_audit
from src.data.checkpoints import CheckpointManager

cfg = load_config(str(PROJECT_ROOT / "configs"))
paths = resolve_paths(cfg)
con = get_duckdb_connection(temp_dir=paths["temp_dir"], **{k: cfg["runtime"]["duckdb"][k] for k in ("memory_limit", "threads")})
ckpt_mgr = CheckpointManager(manifest_path=paths["checkpoints"] / "checkpoint_manifest.json")

staging_dir = paths["interim"] / "staging_shards"
agg_shards = sorted(staging_dir.rglob("agg_*.parquet"))
if not agg_shards:
    raise FileNotFoundError(
        f"Không tìm thấy aggregate Parquet trong {staging_dir}. "
        "Hãy chạy xong notebook 01 với cùng PUBG_STORAGE_MODE và PUBG_DRIVE_PROJECT_ROOT."
    )
print(f"Đầu vào stage 02: {len(agg_shards)} aggregate shards từ {staging_dir}")

In [ ]:
# 1. Làm sạch sơ bộ và ghi removal log
cleaned_pq = paths["interim"] / "cleaned_aggregate.parquet"
removal_csv = paths["tables"] / "removal_log.csv"
clean_summary = audit_and_clean_aggregate_data(con, agg_shards, cleaned_pq, removal_csv)
print(f"Làm sạch: Giữ lại {clean_summary['clean_rows']} dòng hợp lệ.")

In [ ]:
# 2. Xây dựng Match Metadata (N_teams, duration proxy)
meta_pq = paths["interim"] / "match_metadata.parquet"
total_matches = build_match_metadata(con, cleaned_pq, meta_pq)

In [ ]:
# 3. Đánh giá Chronology Grade
meta_df = con.execute(f"SELECT * FROM read_parquet('{str(meta_pq).replace(chr(92), '/')}') LIMIT 50000;").df()
chrono_report = run_chronology_audit(meta_df)
print(f"Chronology Grade: {chrono_report['grade']} - {chrono_report.get('description', chrono_report.get('reason'))}")
atomic_write_json(paths["manifests"] / "chronology_report.json", chrono_report)

In [ ]:
# 4. Khóa Split Assignments (Match isolation invariant)
split_pq = paths["interim"] / "split_assignments.parquet"
split_manifest = paths["manifests"] / "split_manifest.json"
split_meta = create_split_assignments(con, meta_pq, split_pq, split_manifest, strategy="group_by_match")

ckpt_mgr.commit("split_manifest", split_meta["config_hash"], {"split_assignments": split_pq, "match_metadata": meta_pq})
print("Gate G2 Hoàn tất: Split manifest đã được khóa trước EDA quan hệ.")